# Synthetic Axon Bundles

I'm experimenting here with generating synthetic data, specifically, segmentation images of simulated axon bundles.
This should allow us to train on a lot more data, with known ground-truth, so we can see how well different approaches
work on this difficult agglomeration problem.  We can even introduce things like gaps, to test our gap-crossing.

In [ ]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
# Parameters
num_steps = 100  # Minimum number of steps in a path
step_size = 0.03  # How far the axon moves each step
momentum_factor = 0.95  # How much the target's movement direction persists
random_factor = 0.05  # Random variation added to the target's movement

In [ ]:
def grow_axon():
    """Add one axon through our volume.  Return its path as a N x 3 matrix,
    where each row is an [x, y, z] point along the path."""
    # Initialize the starting position on the X=0 face
    axon_position = np.array([0, np.random.uniform(0, 1), np.random.uniform(0, 1)])
    # Initialize the target position outside the cube (X > 1, Y and Z in range [-1, 2])
    target_position = np.array([np.random.uniform(1, 2), np.random.uniform(0, 1), np.random.uniform(0,1)])
    
    # Initialize the target's velocity (momentum) as a small random vector
    target_velocity = np.random.uniform(-0.1, 0.1, 3)
    
    # List of points along the path
    path = [axon_position]
    
    # Simulation loop
    while len(path) < num_steps and axon_position[0] < 1:
        # Compute direction vector from axon to target
        direction = target_position - axon_position
        direction /= np.linalg.norm(direction)  # Normalize to get unit direction
        
        # Move axon toward the target by a small step
        axon_position = axon_position + step_size * direction
        path.append(axon_position)
        
        # Update the target's position with some momentum and random wandering
        target_velocity = momentum_factor * target_velocity + random_factor * np.random.uniform(-0.1, 0.1, 3)
        target_position += target_velocity
        if target_position[0] < 1: target_position[0] = 1

    return np.vstack(path)

In [ ]:
# Grow some axons across the volume
qty_axons = 100
axon_paths = []
for _ in range(qty_axons):
    axon_paths.append(grow_axon())

In [ ]:
# Create a 3D plot
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

for path in axon_paths:
    ax.plot(path[:,0], path[:,1], path[:,2])

ax.set_xlim(0,1)
ax.set_ylim(0,1)
ax.set_zlim(0,1)
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')

plt.show()